# All models 尝试包含全部模型

包括:

* Rolling Normal；
* Linear Quantile；
* Fused K1；
* Fused K3；
* Structured K3；


# 3MDN Model: Structured Mixture Density Network 


本版采用以下数据流程：

1. 直接读取 `1DataDownload_Clean.ipynb` 生成的两张表，不重复生成 target、momentum 或 rolling features。
2. Rolling Normal 先使用原始 `micro_df` 计算，使用截至预测月（含当月）的最近 192 个日历月。
3. Micro 使用显式特征清单；`target_ret_final` 只作为标签，不进入特征清单。
4. Macro 在初始训练期拟合尺度，之后只使用每个时间块开始前的历史月份更新尺度。
5. 自动识别 local、Colab 和 Kaggle；`quick` 模式仅训练两个 epoch，并运行前两个 OOS 窗口。
6. Linear Quantile、Fused 和 Structured 模型在每个窗口重新初始化，不使用 warm start。

## Planned Robustness Checks

1. Normal versus Student-$t$ components.
2. $K=1$ versus multiple mixture components.
3. Rank versus normal-score micro normalization.
4. With and without the macro/FiLM channel.
5. Rolling versus expanding windows and uniform versus time-decay weights.

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from datetime import datetime
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

env = 'local'
# env = 'colab'
# env = 'kaggle'

if env == 'kaggle':
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/MacroFactor_McCracken.parquet')
    path = '/kaggle/working'

elif env == 'colab':
    import google.colab 
    # 1. 设置环境变量
    google.colab.drive.mount('/content/drive')
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/MacroFactor_McCracken.parquet') # 可以根据需要切换到 Colab 的路径
    path = '/content/drive/MyDrive/Colab Notebooks/'


elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet') # local 路径
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")


In [2]:
from pathlib import Path
import importlib.util
import copy
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, t as student_t

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

random_seed = 42
seed_everything(random_seed)


Using device: mps
Global seed set to 42 to ensure reproducibility.


In [3]:
# ==========================================
# 1. 全局参数设置
# ==========================================
run_test = 'quick'  # 'quick' 或 'full'
random_seed = 42

initial_train_months = 180
val_months = 12
test_months = 12

# 每个窗口独立初始化，并使用 Validation early stopping。
early_stopping_max_epochs = 300
batch_size = 1024
lr = 3e-3
patience = 30
dropout_rate = 0.05
sigma_floor = 0.005  # 0.5% monthly component-scale floor

normalization_eps = 1e-8

windows_type = 'rolling'      # 'expanding' 或 'rolling'
weights_type = 'month_equal'  # 'uniform', 'month_equal', or 'time_decay'

ALPHAS = (0.01, 0.05, 0.10)
ALPHA_SUFFIX = {0.01: "1", 0.05: "5", 0.10: "10"}

QUANTILE_LEVELS = np.array([
    0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
    0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.9975, 0.999
], dtype=float)

In [4]:
print('Micro DataFrame shape:', micro_df.shape)
print('Micro months:', micro_df['mthcaldt'].nunique())
display(pd.concat([micro_df.head(5), micro_df.tail(5)]))
print("Macro DataFrame shape:", macro_df.shape)
display(pd.concat([macro_df.head(5), macro_df.tail(5)]))


Micro DataFrame shape: (1560360, 41)
Micro months: 408


,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49,target_ret_final,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,0.013158,9.625,1.014475e+04,4920.0,012994,0.896385,5.690808,3.307560,0.445982,4.872598,14.070388,0.044401,0.099749,0.138304,0.082956,0.112235,0.112708,0.161306,0.392076,2.430626,0.061202,0.672101,1.796365,0.316439,1.340247,1.166318,8.789413,10.931698,3.261452,0.000000,0.0,0.0,-0.055775,31,0.012987,0.009425,0.002110,0.022682,9.224712
1,10001,1991-02-28,0.012987,9.750,1.027650e+04,4920.0,012994,0.975332,5.299653,3.037383,0.434947,5.205927,14.253121,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,-0.012244,0.029179,0.003717,0.022716,9.237615
2,10001,1991-03-31,-0.012244,9.500,1.001300e+04,4920.0,012994,0.975332,5.299653,2.959502,0.423795,5.072442,13.887656,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.039474,0.029348,0.001628,0.022954,9.211640
3,10001,1991-04-30,0.039474,9.875,1.040825e+04,4920.0,012994,0.975332,5.299653,3.115265,0.446100,5.339412,14.618585,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.000000,0.016744,0.004917,0.025398,9.250354
4,10001,1991-05-31,0.000000,9.875,1.040825e+04,4920.0,012994,1.040348,5.316239,3.086420,0.444182,6.893394,14.618585,0.047832,0.103923,0.139492,0.083400,0.119963,0.115336,0.165372,0.375721,2.271979,0.059539,0.472059,1.708091,0.318617,1.389858,1.185531,8.331080,10.836074,3.210526,0.000000,0.0,0.0,-0.019640,31,0.079241,0.070429,0.005972,0.024859,9.250354
1560355,93436,2024-08-31,-0.077390,214.110,6.840044e+08,3711.0,184996,0.105166,57.713252,59.310249,7.174588,59.301712,335.706250,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.221942,-0.100783,-0.006184,0.139169,20.343475
1560356,93436,2024-09-30,0.221942,261.630,8.390474e+08,3711.0,184996,0.105166,57.713252,72.473684,8.802612,72.758187,346.216280,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,-0.045025,-0.144312,0.014849,0.153502,20.547778
1560357,93436,2024-10-31,-0.045025,249.850,8.020335e+08,3711.0,184996,0.105166,57.713252,69.210526,8.406271,69.482219,369.690039,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.381469,0.302679,0.027542,0.140071,20.502661
1560358,93436,2024-11-30,0.381469,345.160,1.107984e+09,3711.0,184996,0.083346,53.762150,93.286486,11.394010,76.450592,306.628288,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.170008,0.040696,0.043050,0.167882,20.825808
1560359,93436,2024-12-31,0.170008,403.840,1.298958e+09,3711.0,184996,0.083346,53.762150,109.145946,13.368497,89.698836,274.076290,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.001882,0.389088,0.054301,0.171772,20.984828


Macro DataFrame shape: (408, 20)


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-0.001000,0.1,0.023836,-0.005920,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.000930,-0.50,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
1,1991-02-28,-0.001892,0.1,-0.041536,-0.004931,-0.003955,-0.029746,0.02,6.682109,0.001999,-0.093898,-0.007471,-0.40,-0.70,-0.3,-0.000014,26.9318,3.54,1.87,2.36
2,1991-03-31,-0.002384,0.2,0.091651,-0.007933,-0.009491,0.005330,0.02,6.872128,0.000704,0.100378,0.005295,-0.66,-0.61,0.5,-0.002975,22.1126,3.82,1.91,2.22
3,1991-04-30,-0.001568,0.2,0.024146,-0.009247,0.006162,-0.061953,0.00,6.825460,-0.000088,0.043726,-0.002343,-0.13,-0.08,0.2,-0.000742,19.0410,3.97,2.20,1.98
4,1991-05-31,-0.001709,-0.1,-0.082863,0.003497,0.011971,0.041812,0.00,6.908755,-0.002301,0.034535,-0.002986,-0.21,-0.34,0.6,0.002221,18.9272,4.03,2.39,1.90
403,2024-08-31,0.000718,0.2,0.006105,-0.006401,0.003767,0.094333,0.00,7.121252,-0.002100,-0.024829,-0.006629,0.00,0.03,-0.1,0.002110,14.4084,0.51,-0.95,1.59
404,2024-09-30,0.000895,-0.1,-0.030903,0.008110,0.001463,0.000459,-0.01,7.212294,0.003887,0.030865,0.003711,0.00,-0.19,0.4,0.000322,19.6750,0.27,-1.18,1.73
405,2024-10-31,0.001598,-0.1,-0.029656,-0.002832,0.003543,-0.007867,0.01,7.210818,-0.001050,-0.031354,0.002598,-0.20,-0.26,1.6,-0.000073,17.6597,0.29,-1.00,1.70
406,2024-11-30,0.000075,0.0,0.055299,-0.002629,0.001225,0.002425,0.00,7.178545,0.000319,0.023649,-0.000157,-0.30,-0.24,0.3,0.000641,19.9478,0.80,-0.41,1.53
407,2024-12-31,0.001426,0.1,-0.078988,-0.001472,0.002754,-0.010575,0.00,7.161622,0.002139,0.024533,-0.002535,-0.19,-0.09,0.8,0.000686,15.9822,1.13,-0.06,1.42


### ==========================================
### Rolling Normal：使用截至预测月的最近 192 个日历月
### ==========================================

In [5]:
ROLLING_SCALE_FLOOR = 1e-6


def normal_var_and_loss_es(alpha, pred_mean, pred_vol):
    """计算 Normal distribution 的左尾 VaR 和 loss-ES。"""
    z_alpha = norm.ppf(alpha)
    var_value = pred_mean + pred_vol * z_alpha
    es_return = pred_mean - pred_vol * norm.pdf(z_alpha) / alpha
    return float(var_value), float(-es_return)


def run_rolling_normal(panel, max_history_months):
    """
    对每只股票使用截至预测月的最近 max_history_months 个日历月估计 Normal distribution。
    历史窗口包含预测月；股票缺月时不会用更早的观测补足 192 条。
    """
    panel = panel.sort_values(["permno", "mthcaldt"]).reset_index(drop=True).copy()
    panel["mthcaldt"] = pd.to_datetime(panel["mthcaldt"]) + pd.offsets.MonthEnd(0)

    # 将日期转为连续月份编号，便于严格按日历月确定每只股票的左边界。
    month_id = panel["mthcaldt"].dt.year.to_numpy() * 12 + panel["mthcaldt"].dt.month.to_numpy()
    returns = panel["mthret"].to_numpy(dtype=float)
    pred_mean = np.full(len(panel), np.nan)
    pred_vol = np.full(len(panel), np.nan)
    history_n = np.zeros(len(panel), dtype=np.int32)

    for positions in panel.groupby("permno", sort=False).indices.values():
        positions = np.asarray(positions)
        stock_months = month_id[positions]
        stock_returns = returns[positions]

        # 对月份 t，使用 [t-max_history_months+1, t] 的收益率，共 max_history_months 个日历月。
        left = np.searchsorted(stock_months, stock_months - max_history_months + 1, side="left")
        right = np.arange(len(positions)) + 1  # 包含当月
        count = right - left

        cumulative_sum = np.r_[0.0, np.cumsum(stock_returns)]
        cumulative_sq_sum = np.r_[0.0, np.cumsum(stock_returns ** 2)]
        window_sum = cumulative_sum[right] - cumulative_sum[left]
        window_sq_sum = cumulative_sq_sum[right] - cumulative_sq_sum[left]

        has_mean = count > 0
        has_vol = count > 1
        pred_mean[positions[has_mean]] = window_sum[has_mean] / count[has_mean]
        variance = (window_sq_sum[has_vol] - window_sum[has_vol] ** 2 / count[has_vol]) / (count[has_vol] - 1)
        pred_vol[positions[has_vol]] = np.sqrt(np.maximum(variance, 0.0))
        history_n[positions] = count

    panel["pred_mean"] = pred_mean
    panel["pred_vol"] = pred_vol
    panel["history_n"] = history_n

    # 新上市股票只有当月收益时，均值使用该收益，波动率使用当月横截面标准差。
    monthly_fallback = panel.groupby("mthcaldt")["mthret"].agg(["mean", "std"])
    panel["pred_mean"] = panel["pred_mean"].fillna(panel["mthcaldt"].map(monthly_fallback["mean"]))
    panel["pred_vol"] = panel["pred_vol"].fillna(panel["mthcaldt"].map(monthly_fallback["std"]))
    panel["pred_vol"] = panel["pred_vol"].clip(lower=ROLLING_SCALE_FLOOR)

    available_dates = pd.DatetimeIndex(np.sort(panel["mthcaldt"].unique()))
    oos_panel = panel[panel["mthcaldt"] >= available_dates[max_history_months]].copy()
    if run_test == "quick":
        quick_dates = np.sort(oos_panel["mthcaldt"].unique())[:2 * test_months]
        oos_panel = oos_panel[oos_panel["mthcaldt"].isin(quick_dates)].copy()

    output_rows = []
    for _, row in tqdm(oos_panel.iterrows(), total=len(oos_panel), desc="Rolling Normal"):
        output_row = {
            "permno": row["permno"], "mthcaldt": row["mthcaldt"],
            "target_ret_final": float(row["target_ret_final"]), "model_id": "rolling_normal",
            "pred_mean": float(row["pred_mean"]), "pred_vol": float(row["pred_vol"]),
            "history_n": int(row["history_n"]),
        }
        for alpha in ALPHAS:
            suffix = ALPHA_SUFFIX[alpha]
            var_value, es_loss = normal_var_and_loss_es(alpha, output_row["pred_mean"], output_row["pred_vol"])
            output_row[f"var_{suffix}"] = var_value
            output_row[f"es_{suffix}"] = es_loss
        output_rows.append(output_row)

    return pd.DataFrame(output_rows).sort_values(["mthcaldt", "permno"]).reset_index(drop=True)


rolling_normal_forecasts = run_rolling_normal(
    panel=micro_df, max_history_months=initial_train_months + val_months
)

Rolling Normal:   0%|          | 0/87375 [00:00<?, ?it/s]

In [6]:
rolling_normal_forecasts.to_parquet(
    os.path.join(path, f'rolling_normal_forecasts_{timestamp}.parquet'), index=False
)

display(pd.concat([rolling_normal_forecasts.head(4), rolling_normal_forecasts.tail(4)]))

rolling_normal_summary = rolling_normal_forecasts.groupby("mthcaldt").agg(
    n_firms=("permno", "size"), mean_history_n=("history_n", "mean"),
    mean_predicted_vol=("pred_vol", "mean"), mean_es_5=("es_5", "mean"),
)
display(rolling_normal_summary.head())

,permno,mthcaldt,target_ret_final,model_id,pred_mean,pred_vol,history_n,var_1,es_1,var_5,es_5,var_10,es_10
0,10001,2007-01-31,0.264774,rolling_normal,0.011201,0.083830,192,-0.183818,0.212225,-0.126688,0.161717,-0.096232,0.135920
1,10002,2007-01-31,0.033054,rolling_normal,0.015754,0.078892,186,-0.167775,0.194509,-0.114011,0.146977,-0.085350,0.122699
2,10025,2007-01-31,-0.028094,rolling_normal,0.023622,0.155724,192,-0.338647,0.391417,-0.232522,0.297593,-0.175947,0.249672
3,10026,2007-01-31,-0.038517,rolling_normal,0.017167,0.099552,192,-0.214426,0.248161,-0.146582,0.188181,-0.110415,0.157546
87371,92869,2008-12-31,-0.366006,rolling_normal,-0.262308,0.179347,1,-0.679532,0.740307,-0.557308,0.632250,-0.492151,0.577059
87372,92872,2008-12-31,-0.175000,rolling_normal,0.000000,0.179347,1,-0.417224,0.477999,-0.295000,0.369942,-0.229843,0.314751
87373,92874,2008-12-31,-0.496259,rolling_normal,0.030398,0.150084,135,-0.318749,0.369607,-0.216468,0.279182,-0.161942,0.232996
87374,93105,2008-12-31,0.106070,rolling_normal,0.024958,0.109264,173,-0.229228,0.266254,-0.154765,0.200422,-0.115069,0.166798


,n_firms,mean_history_n,mean_predicted_vol,mean_es_5
mthcaldt,,,,
2007-01-31,4041,112.355110,0.129350,0.242610
2007-02-28,4040,112.580693,0.128467,0.242007
2007-03-31,4016,112.689243,0.128315,0.241911
2007-04-30,4025,112.450186,0.127447,0.240163
2007-05-31,4026,112.165425,0.127308,0.239549


In [7]:
# =========================================================
# 1. Validate the cleaned micro table and normalize explicit features
# =========================================================
micro_normalize = 'normal'  # 'normal' or 'rank' for accounting-ratio features

micro_df = micro_df.copy()

micro_exclude_cols = {
    'permno', 'mthcaldt', 'mthcap', 'mthprc', 'siccd', 'gvkey', 'ffi49', 'target_ret_final'
}

micro_cols = [
    col for col in micro_df.columns
    if col not in micro_exclude_cols
]

micro_df = micro_df[['permno', 'mthcaldt', 'ffi49', 'target_ret_final'] + micro_cols].copy()
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)

if micro_normalize == 'rank':
    micro_norm = (
        micro_df.groupby('mthcaldt')[micro_cols].rank(pct=True) * 2.0 - 1.0
    )
elif micro_normalize == 'normal':
    monthly_mean = micro_df.groupby('mthcaldt')[micro_cols].transform('mean')
    monthly_std = micro_df.groupby('mthcaldt')[micro_cols].transform('std')
    micro_norm = (
        micro_df[micro_cols] - monthly_mean
    ) / (monthly_std + normalization_eps)
else:
    raise ValueError("micro_normalize must be 'normal' or 'rank'")
micro_df[micro_cols] = micro_norm.fillna(0.0).to_numpy()

micro_df = micro_df.sort_values(['mthcaldt', 'permno']).reset_index(drop=True)

print('Micro model panel:', micro_df.shape)

display(pd.concat([micro_df.head(3), micro_df.tail(3)]))

Micro model panel: (1560360, 37)


,permno,mthcaldt,ffi49,target_ret_final,mthret,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,31,0.012987,-0.482931,-0.060272,-0.008572,-0.346230,-0.261481,-0.094552,0.117493,0.022348,0.022290,0.000135,0.019851,-0.022173,-0.012717,-0.673646,0.807587,-0.025520,-0.802450,0.094626,-0.107216,-0.182591,-0.142714,0.006846,-0.075379,0.014827,-0.007216,-0.158612,-0.248521,-0.379298,-0.248863,0.076749,-0.130990,-1.658133,-1.454875
1,10009,1991-01-31,0,0.038462,-0.697982,-0.150198,-0.003516,-0.050630,-0.201273,-0.047498,-0.024759,0.023051,0.022233,0.018286,0.021075,-0.022191,-0.017928,-0.159436,-0.083883,-0.042677,-0.330347,-0.037394,-0.186992,-0.210240,-0.077440,-0.095277,-0.122830,0.001247,-0.078805,-0.158612,-0.248521,-0.379298,-0.020074,0.294876,0.063363,-0.770032,-1.733802
2,10010,1991-01-31,12,0.058824,1.528051,-1.098060,0.023959,1.624295,0.394577,2.524322,2.121101,0.023846,0.022293,0.038728,0.019352,-0.014604,0.019690,1.690779,-0.693315,-0.064559,1.005279,-0.157773,-0.296600,-0.230529,0.005193,0.758633,-0.143224,0.057588,-0.070947,0.031733,-0.248521,-0.379298,0.661589,8.417659,6.120281,4.790800,-0.256213
1560357,93426,2024-12-31,37,-0.007243,0.159278,0.658723,-0.020134,0.180521,-0.146922,-0.038097,0.051845,0.019730,0.020026,0.052540,0.019696,0.071010,0.085741,0.323659,-0.646617,-0.032065,0.704358,-0.009179,-0.016124,-0.130097,-0.018407,0.043098,-0.079408,0.027635,-0.100876,-0.019677,-0.317908,-0.066250,0.134618,-0.624502,-0.618882,-0.209830,-0.974859
1560358,93434,2024-12-31,1,0.147685,0.431762,5.490732,-0.045637,-0.332684,-0.153241,-0.315872,-0.228740,0.019539,0.019905,0.049022,0.019495,-0.441455,-0.405977,-0.491370,0.389375,-0.070470,3.022603,-0.034498,-0.004524,-0.363371,-0.020110,-0.507478,-0.115962,-0.246834,-0.108403,-0.019670,-0.317908,-0.066250,-0.586480,-0.764274,0.538531,2.098632,-2.360453
1560359,93436,2024-12-31,23,0.001882,0.521337,-0.868116,0.041442,1.378737,-0.002125,1.785242,4.099359,0.019755,0.020023,0.048992,0.019722,0.226913,0.277166,-0.080058,-0.739260,-0.034645,0.381296,-0.005834,-0.013096,-0.143984,-0.019482,0.303039,0.087998,-0.167611,-0.079103,-0.019684,-0.317908,-0.066250,0.289431,0.068798,0.414510,0.126864,3.161670


In [8]:
# =========================================================
# 2. Macro：初始训练期统一拟合，之后按时间块更新
# =========================================================
def standardize_macro(
    macro_data,
    macro_cols,
    model_dates,
    initial_months,
    update_months,
):
    macro = macro_data.set_index('mthcaldt').loc[model_dates, macro_cols].copy()
    macro_z = pd.DataFrame(index=model_dates, columns=macro_cols, dtype=float)

    initial_dates = model_dates[:initial_months]
    initial_mean = macro.loc[initial_dates].mean()
    initial_std = macro.loc[initial_dates].std(ddof=0)
    macro_z.loc[initial_dates] = (
        (macro.loc[initial_dates] - initial_mean) /
        (initial_std + normalization_eps)
    ).to_numpy()

    for start in range(initial_months, len(model_dates), update_months):
        # print(f'>>> Standardizing Macro features for block starting at {model_dates[start]}...')
        block_dates = model_dates[start:start + update_months]
        history = macro.iloc[:start]
        history_mean = history.mean()
        history_std = history.std(ddof=0)
        macro_z.loc[block_dates] = (
            (macro.loc[block_dates] - history_mean) /
            (history_std + normalization_eps)
        ).to_numpy()

    macro_z.index.name = 'mthcaldt'
    return macro_z.reset_index()

macro_cols = [
    col for col in macro_df.columns
    if col != 'mthcaldt'
]

macro_df['mthcaldt'] = pd.to_datetime(macro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df = macro_df.sort_values('mthcaldt').reset_index(drop=True)

model_dates = pd.DatetimeIndex(np.sort(micro_df['mthcaldt'].unique()))

macro_df = standardize_macro(
    macro_data=macro_df,
    macro_cols=macro_cols,
    model_dates=model_dates,
    initial_months=initial_train_months,
    update_months=test_months,
)
pd.concat([macro_df.head(5), macro_df.tail(5)])

,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1,1991-02-28,-2.192501,0.729992,-0.817965,-1.469064,-1.649264,-1.007572,1.775543,-3.404600,0.550763,-1.445587,-1.227084,-1.945786,-3.240398,-0.599735,0.013918,0.924305,-0.184668,-0.058573,0.470548
2,1991-03-31,-2.549766,1.417942,1.947638,-2.076241,-3.018345,0.041438,1.775543,-2.399603,0.198073,1.546106,0.850978,-3.281213,-2.811241,0.849289,-1.146876,0.257253,0.008029,-0.022803,0.211304
3,1991-04-30,-1.957019,1.417942,0.545911,-2.342198,0.852805,-1.970789,0.173549,-2.646426,-0.017617,0.673709,-0.392313,-0.558996,-0.283985,0.305905,-0.271394,-0.167903,0.111260,0.236528,-0.233113
4,1991-05-31,-2.059530,-0.645909,-1.676114,0.236066,2.289584,1.132493,0.173549,-2.205888,-0.620199,0.532180,-0.496941,-0.969897,-1.523771,1.030417,0.890301,-0.183655,0.152552,0.406434,-0.381253
403,2024-08-31,-0.019744,0.360853,0.054717,-0.858590,0.157267,2.364269,0.067085,-0.036957,-0.404763,-0.034278,-0.618382,0.033549,0.170575,-0.106644,0.687854,-0.684488,-2.068622,-2.216100,-1.064560
404,2024-09-30,0.002850,-0.163222,-0.189031,0.681921,-0.066753,-0.049609,-0.415929,0.225697,0.730580,0.042472,0.346484,0.033549,-0.915719,0.336030,0.108108,-0.033047,-2.216690,-2.410739,-0.868202
405,2024-10-31,0.092972,-0.163222,-0.180820,-0.479714,0.135471,-0.263698,0.550100,0.221439,-0.205664,-0.043270,0.242691,-1.037861,-1.261358,1.398448,-0.020236,-0.282325,-2.204351,-2.258413,-0.910279
406,2024-11-30,-0.102179,0.011470,0.378732,-0.458106,-0.089945,0.000934,0.067085,0.128332,0.053851,0.032528,-0.014406,-1.573566,-1.162604,0.247495,0.211272,0.000696,-1.889707,-1.759122,-1.148713
407,2024-12-31,0.070973,0.186161,-0.505745,-0.335332,0.058751,-0.333345,0.067085,0.079508,0.399053,0.033746,-0.236290,-0.984291,-0.421949,0.690169,0.226152,-0.489820,-1.686113,-1.462933,-1.302994


In [9]:
# =========================================================
# 3. 最后合并标准化后的 Micro 和 Macro
# =========================================================
processed_df = (
    micro_df
    .merge(macro_df, on='mthcaldt', how='left', validate='many_to_one')
    .sort_values(['mthcaldt', 'permno'])
    .reset_index(drop=True)
)

print(f'Data preprocessing complete. Final shape: {processed_df.shape}')
print(f'Micro features: {len(micro_cols)}; Macro features: {len(macro_cols)}')

pd.concat([processed_df.head(5), processed_df.tail(5)])

Data preprocessing complete. Final shape: (1560360, 56)
Micro features: 33; Macro features: 19


,permno,mthcaldt,ffi49,target_ret_final,mthret,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,10001,1991-01-31,31,0.012987,-0.482931,-0.060272,-0.008572,-0.346230,-0.261481,-0.094552,0.117493,0.022348,0.022290,0.000135,0.019851,-0.022173,-0.012717,-0.673646,0.807587,-0.025520,-0.802450,0.094626,-0.107216,-0.182591,-0.142714,0.006846,-0.075379,0.014827,-0.007216,-0.158612,-0.248521,-0.379298,-0.248863,0.076749,-0.130990,-1.658133,-1.454875,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1,10009,1991-01-31,0,0.038462,-0.697982,-0.150198,-0.003516,-0.050630,-0.201273,-0.047498,-0.024759,0.023051,0.022233,0.018286,0.021075,-0.022191,-0.017928,-0.159436,-0.083883,-0.042677,-0.330347,-0.037394,-0.186992,-0.210240,-0.077440,-0.095277,-0.122830,0.001247,-0.078805,-0.158612,-0.248521,-0.379298,-0.020074,0.294876,0.063363,-0.770032,-1.733802,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
2,10010,1991-01-31,12,0.058824,1.528051,-1.098060,0.023959,1.624295,0.394577,2.524322,2.121101,0.023846,0.022293,0.038728,0.019352,-0.014604,0.019690,1.690779,-0.693315,-0.064559,1.005279,-0.157773,-0.296600,-0.230529,0.005193,0.758633,-0.143224,0.057588,-0.070947,0.031733,-0.248521,-0.379298,0.661589,8.417659,6.120281,4.790800,-0.256213,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
3,10016,1991-01-31,34,0.161039,-0.821051,0.529053,-0.011430,-0.297854,-0.322713,-2.263505,-0.179953,0.021483,0.022133,0.014085,0.018187,-0.016929,0.000100,0.047139,0.824841,-0.025595,-0.532339,-0.089922,-0.054475,-0.164662,-0.077783,0.166408,-0.132934,0.033594,0.051550,-0.094274,-0.248521,-0.379298,0.956625,-0.911423,-1.306998,-0.043820,0.131166,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
4,10019,1991-01-31,38,0.076923,-0.572769,0.008433,-0.006695,-0.246446,-0.204817,-0.090451,-0.022912,0.031221,0.029770,0.028113,0.027323,-0.003052,-0.070490,0.776617,-0.834677,-0.070709,-0.871221,0.558000,-0.331307,0.343498,0.322973,0.222478,-0.131284,0.189505,-0.115895,0.338476,-0.248521,-0.379298,0.153135,-1.097026,-1.483328,0.147944,-0.763204,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1560355,93374,2024-12-31,46,0.012492,-0.144992,0.287873,-0.007011,0.831951,-0.145354,0.015344,0.124609,0.019723,0.020011,0.046602,0.019685,0.051502,0.077170,-0.654780,-0.526210,-0.006783,0.554310,-0.023671,0.003959,-0.238241,-0.019852,-0.444575,-0.092518,-0.111928,-0.091199,-0.019701,-0.317908,-0.066250,0.204695,-0.192988,-0.238248,-0.259101,0.553505,0.070973,0.186161,-0.505745,-0.335332,0.058751,-0.333345,0.067085,0.079508,0.399053,0.033746,-0.236290,-0.984291,-0.421949,0.690169,0.226152,-0.489820,-1.686113,-1.462933,-1.302994
1560356,93397,2024-12-31,1,-0.057236,-0.180626,-0.265380,0.175037,0.705630,-0.131255,0.302743,0.621994,0.019730,0.019991,0.047888,0.019694,0.069142,0.073307,-0.407801,-0.557938,0.035982,-0.548790,-0.005080,-0.015178,-0.346834,-0.020266,-0.041519,-0.044009,-0.097204,-0.100247,-0.019701,-0.317908,-0.066250,0.169230,0.042248,-0.045523,-0.258912,-0.761835,0.07097

### ==========================================
### Linear Quantile：线性分位数回归
### ==========================================

In [10]:
linear_batch_size = 8192
linear_lr = 1e-3
linear_weight_decay = 1e-4

In [11]:
def tensor_batch_indices(length, batch_size, device, shuffle=True):
    if shuffle:
        indices = torch.randperm(length, device=device)
    else:
        indices = torch.arange(length, device=device)
    for start in range(0, length, batch_size):
        yield indices[start:start + batch_size] 

In [12]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col

    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months  < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months < len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

def make_window_generator(panel):
    generator = DataWindowGenerator(df=panel, date_col="mthcaldt")

    if windows_type == "expanding":
        return generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    return generator.rolling_window_generator(
        train_months=initial_train_months,
        val_months=val_months,
        test_months=test_months,
    )

In [13]:
# ==========================================
# Linear Quantile 模型
# Macro + Micro + FF49 行业固定效应
# ==========================================
class LinearQuantile(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, quantile_levels):
        super().__init__()
        self.num_industries = num_industries
        input_dim = macro_dim + micro_dim + num_industries - 1
        self.linear = nn.Linear(input_dim, len(quantile_levels))

        quantiles = torch.tensor(quantile_levels, dtype=torch.float32).view(1, -1)
        self.register_buffer("quantile_levels", quantiles)

    def forward(self, macro, micro, industry):
        industry_dummy = F.one_hot(
            industry.long(), num_classes=self.num_industries
        ).to(macro.dtype)[:, 1:]

        x = torch.cat([macro, micro, industry_dummy], dim=1)
        return self.linear(x)

In [14]:
def weighted_pinball_loss(pred, target, weight, quantile_levels):
    target = target.reshape(-1, 1)
    weight = weight.reshape(-1)

    error = target - pred
    loss = torch.maximum(
        quantile_levels * error,
        (quantile_levels - 1.0) * error,
    )
    loss_per_stock = loss.mean(dim=1)

    return (
        (loss_per_stock * weight).sum()
        / weight.sum().clamp_min(1e-12)
    )

In [15]:
# ==========================================
# Sample weights: prevent months with more listed stocks from dominating
# ==========================================
def calculate_sample_weights(df_dates, mode='month_equal', half_life_months=36):
    dates = pd.to_datetime(df_dates).reset_index(drop=True)
    if mode == 'uniform':
        weights = np.ones(len(dates), dtype=np.float64)
    elif mode in {'month_equal', 'time_decay'}:
        month_counts = dates.groupby(dates).transform('size').to_numpy(dtype=float)
        weights = 1.0 / np.maximum(month_counts, 1.0)
        if mode == 'time_decay':
            max_date = dates.max()
            months_diff = (
                (max_date.year - dates.dt.year) * 12
                + (max_date.month - dates.dt.month)
            ).to_numpy(dtype=float)
            weights *= np.power(0.5, months_diff / half_life_months)
    else:
        raise ValueError("weights_type must be 'uniform', 'month_equal', or 'time_decay'")
    weights = weights / weights.mean()
    return weights.astype(np.float32)

# ==========================================
# 将完整面板转成设备端 Tensor
# ==========================================
def make_device_tensors(panel):
    X_macro = torch.from_numpy(
        panel[macro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    X_micro = torch.from_numpy(
        panel[micro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    industry = torch.from_numpy(
        panel["ffi49"].to_numpy(dtype=np.int64, copy=True)
    ).to(device)

    target = torch.from_numpy(
        panel["target_ret_final"].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    return X_macro, X_micro, industry, target

In [16]:
# ==========================================
# Linear Quantile 训练
# ==========================================
def train_linear_quantile(
    model,
    X_train_macro, X_train_micro, X_train_industry, y_train, w_train,
    X_val_macro, X_val_micro, X_val_industry, y_val, w_val,
    epochs=300, batch_size=8192, lr=1e-3, patience=20,
):
    if run_test == "quick":
        epochs = 2
        patience = 1

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=linear_weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
        mode="min", factor=0.5, patience=5, threshold=1e-4, threshold_mode="abs", min_lr=1e-6,
    )

    train_loss_history = []
    val_loss_history = []
    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    train_size = len(y_train)

    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            train_size,
            batch_size,
            X_train_macro.device,
            shuffle=True,
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx]
            b_w = w_train[batch_idx]

            optimizer.zero_grad()
            pred = model(b_macro, b_micro, b_industry)
            loss = weighted_pinball_loss(
                pred,
                b_y,
                b_w,
                model.quantile_levels,
            )
            loss.backward()
            optimizer.step()

            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight

        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_pred = model(
                X_val_macro,
                X_val_micro,
                X_val_industry,
            )
            val_loss = weighted_pinball_loss(
                val_pred,
                y_val,
                w_val,
                model.quantile_levels,
            ).item()

        if not np.isfinite(val_loss):
            raise RuntimeError("Linear Quantile validation loss became non-finite.")

        val_loss_history.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    return (
        model,
        best_val_loss,
        train_loss_history,
        val_loss_history,
    )

In [17]:
# ==========================================
# 从预测分位数计算均值、波动率、VaR 和 ES
# ==========================================
def add_quantile_distribution_results(output, quantile_pred):
    levels = QUANTILE_LEVELS
    quantile_pred = np.sort(quantile_pred, axis=1)
    trapz = np.trapezoid

    full_levels = np.r_[0.0, levels, 1.0]
    full_quantiles = np.column_stack([
        quantile_pred[:, 0],
        quantile_pred,
        quantile_pred[:, -1],
    ])

    pred_mean = trapz(full_quantiles, full_levels, axis=1)
    second_moment = trapz(full_quantiles ** 2, full_levels, axis=1)

    output["pred_mean"] = pred_mean
    output["pred_vol"] = np.sqrt(
        np.maximum(second_moment - pred_mean ** 2, 0.0)
    )

    for alpha in ALPHAS:
        suffix = ALPHA_SUFFIX[alpha]
        alpha_idx = np.flatnonzero(np.isclose(levels, alpha))[0]
        below_alpha = levels < alpha

        var_value = quantile_pred[:, alpha_idx]
        tail_levels = np.r_[0.0, levels[below_alpha], alpha]
        tail_quantiles = np.column_stack([
            quantile_pred[:, 0],
            quantile_pred[:, below_alpha],
            var_value,
        ])

        output[f"var_{suffix}"] = var_value
        output[f"es_{suffix}"] = (
            -trapz(tail_quantiles, tail_levels, axis=1) / alpha
        )

    output["quantile_levels"] = json.dumps(levels.tolist())
    output["quantile_vec"] = [
        json.dumps(row.tolist()) for row in quantile_pred
    ]

    return output

In [18]:
# ==========================================
# Linear Quantile 滚动样本外预测
# ==========================================
def run_linear_quantile(panel):
    panel = panel.sort_values(["mthcaldt", "permno"]).reset_index(drop=True)
    window_gen = make_window_generator(panel)

    global_X_mac, global_X_mic, global_ind, global_Y = make_device_tensors(panel)
    
    all_predictions = []
    history_logs = []

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )

        start_time = time.time()
        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, Y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        X_va_mic, X_va_ind, Y_va = global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]


        w_tr = torch.from_numpy(
            calculate_sample_weights(
                train_df["mthcaldt"],
                mode=weights_type,
                half_life_months=36,
            )
        ).to(device)

        w_va = torch.from_numpy(
            calculate_sample_weights(
                val_df["mthcaldt"],
                mode=weights_type,
                half_life_months=36,
            )
        ).to(device)

        # 每个窗口重新初始化，不使用 warm start
        seed_everything(random_seed + window_idx)

        model = LinearQuantile(
            macro_dim=len(macro_cols),
            micro_dim=len(micro_cols),
            num_industries=50,
            quantile_levels=QUANTILE_LEVELS,
        ).to(device)

        # 使用训练收益率的无条件分位数初始化截距
        initial_quantiles = np.quantile(
            train_df["target_ret_final"].to_numpy(),
            QUANTILE_LEVELS,
        )

        with torch.no_grad():
            model.linear.weight.zero_()
            model.linear.bias.copy_(
                torch.tensor(
                    initial_quantiles,
                    dtype=torch.float32,
                    device=device,
                )
            )

        model, best_val_loss, tr_hist, va_hist = train_linear_quantile(
            model=model,
            X_train_macro=X_tr_mac,
            X_train_micro=X_tr_mic,
            X_train_industry=X_tr_ind,
            y_train=Y_tr,
            w_train=w_tr,
            X_val_macro=X_va_mac,
            X_val_micro=X_va_mic,
            X_val_industry=X_va_ind,
            y_val=Y_va,
            w_val=w_va,
            epochs=early_stopping_max_epochs,
            batch_size=linear_batch_size,
            lr=linear_lr,
            patience=patience,
        )

        model.eval()
        with torch.no_grad():
            quantile_pred = model(
                X_te_mac,
                X_te_mic,
                X_te_ind,
            ).cpu().numpy()

        output = test_df[
            ["permno", "mthcaldt", "target_ret_final"]
        ].copy()

        output["model_id"] = "linear_quantile"
        output["validation_loss"] = best_val_loss
        output["epochs_run"] = len(tr_hist)

        output = add_quantile_distribution_results(
            output,
            quantile_pred,
        )
        all_predictions.append(output)

        history_logs.append({
            "window": window_idx,
            "train_period": f"{info['train'][0]} to {info['train'][1]}",
            "test_period": f"{info['test'][0]} to {info['test'][1]}",
            "duration_sec": time.time() - start_time,
            "best_val_loss": best_val_loss,
            "best_epoch": int(np.argmin(va_hist) + 1),
            "epochs_run": len(tr_hist),
            "train_loss_trace": tr_hist,
            "val_loss_trace": va_hist,
        })

        if run_test == "quick" and window_idx == 1:
            break

    forecasts = pd.concat(all_predictions, ignore_index=True)
    metrics = pd.DataFrame(history_logs)

    return forecasts, metrics

In [19]:
linear_quantile_forecasts, linear_quantile_metrics = run_linear_quantile(processed_df)
# 执行保存
linear_quantile_forecasts.to_parquet(os.path.join(path, f'linear_quantile_forecasts_{timestamp}.parquet'), index=False)
linear_quantile_metrics.to_parquet(os.path.join(path, f'linear_quantile_metrics_{timestamp}.parquet'), index=False)

display(pd.concat([linear_quantile_forecasts.head(4),linear_quantile_forecasts.tail(4),]))

[Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,validation_loss,epochs_run,pred_mean,pred_vol,var_1,es_1,var_5,es_5,var_10,es_10,quantile_levels,quantile_vec
0,10001,2007-01-31,0.264774,linear_quantile,0.010749,2,-0.000755,0.132999,-0.283056,0.361087,-0.182243,0.249723,-0.131247,0.203234,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.4730873107910156, -0.3945900797843933, -0...."
1,10002,2007-01-31,0.033054,linear_quantile,0.010749,2,0.012649,0.126412,-0.280199,0.370454,-0.180050,0.253481,-0.120001,0.201753,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.4706961214542389, -0.414999783039093, -0.3..."
2,10025,2007-01-31,-0.028094,linear_quantile,0.010749,2,0.017495,0.185713,-0.423751,0.530100,-0.266666,0.368806,-0.185905,0.297546,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.6557271480560303, -0.5786260366439819, -0...."
3,10026,2007-01-31,-0.038517,linear_quantile,0.010749,2,0.007004,0.109156,-0.228254,0.294194,-0.150462,0.204237,-0.103228,0.165541,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.38860613107681274, -0.31636232137680054, -..."
87371,92869,2008-12-31,-0.366006,linear_quantile,0.012217,2,0.038232,0.239871,-0.734491,0.875780,-0.304413,0.539873,-0.186357,0.392629,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.1713635921478271, -0.8610473871231079, -0...."
87372,92872,2008-12-31,-0.175000,linear_quantile,0.012217,2,0.041826,0.271667,-0.788329,0.918998,-0.366499,0.594314,-0.236152,0.447820,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.2190353870391846, -0.8913432359695435, -0...."
87373,92874,2008-12-31,-0.496259,linear_quantile,0.012217,2,0.063024,0.228747,-0.663695,0.792679,-0.266757,0.483470,-0.151488,0.346296,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.0707273483276367, -0.765225887298584, -0.7..."
87374,93105,2008-12-31,0.106070,linear_quantile,0.012217,2,0.074973,0.229690,-0.701190,0.851771,-0.259586,0.504804,-0.137710,0.351726,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.1497821807861328, -0.8347830772399902, -0...."


### ==========================================
### Fused models (K=1 and 3)
### ==========================================

In [20]:
# ==========================================
# Fused MDN
# K=1：Conditional Normal
# K=3：Fused Normal Mixture
# ==========================================
class FusedMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6,
        hidden_dim=64, head_dim=8, num_components=1, dropout_rate=0.05, dist_type="normal",
    ):
        super().__init__()

        self.num_components = num_components
        self.dist_type = dist_type
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        input_dim = macro_dim + micro_dim + ind_emb_dim

        self.fused_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim // 2, head_dim),
            nn.LayerNorm(head_dim),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(head_dim, num_components) if num_components > 1 else None
        self.mu_head = nn.Linear(head_dim, num_components)
        self.sigma_head = nn.Linear(head_dim, num_components)
        self.nu_head = nn.Linear(head_dim, num_components) if dist_type == "t" else None

        # 行业 embedding 不要初始化得太大
        nn.init.normal_(self.ind_embedding.weight, mean=0.0, std=0.05)

        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)

        nn.init.xavier_normal_(self.sigma_head.weight, gain=0.1)
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.pi_head is not None:
            # K=3 初始概率相同
            nn.init.zeros_(self.pi_head.weight)
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            nn.init.xavier_normal_(self.nu_head.weight, gain=0.1)
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_industry):
        industry_embedding = self.ind_embedding(x_industry)

        x = torch.cat([x_macro, x_micro, industry_embedding], dim=1,)
        h = self.fused_extractor(x)

        if self.pi_head is None:
            pi_logits = h.new_zeros((len(h), 1))
        else:
            pi_logits = self.pi_head(h)

        mu = self.mu_head(h)
        sigma = F.softplus(self.sigma_head(h)) + sigma_floor

        if self.nu_head is None:
            nu = None
        else:
            nu = F.softplus(self.nu_head(h)) + 2.01

        return pi_logits, mu, sigma, nu

In [21]:
# ==========================================
# 所有 MDN 共用的加权 NLL
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type="normal",):
    target = target.reshape(-1, 1)
    weights = weights.reshape(-1)

    log_pi = torch.log_softmax(pi_logits, dim=1)

    if dist_type == "normal":
        dist = torch.distributions.Normal(mu, sigma)
    elif dist_type == "t":
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma,)
    else:
        raise ValueError("dist_type must be 'normal' or 't'")

    log_component = dist.log_prob(target.expand_as(mu))
    observation_nll = -torch.logsumexp( log_pi + log_component, dim=1,)

    return (observation_nll * weights).sum() / weights.sum().clamp_min(1e-12)

In [22]:
# ==========================================
# 所有 MDN 共用的 Validation
# ==========================================
def evaluate_mdn(model, X_macro, X_micro, X_industry, target, weights, batch_size,):
    model.eval()
    loss_numerator = 0.0
    weight_sum = 0.0

    with torch.no_grad():
        for batch_idx in tensor_batch_indices(len(target), batch_size, X_macro.device, shuffle=False):
            b_macro = X_macro[batch_idx]
            b_micro = X_micro[batch_idx]
            b_industry = X_industry[batch_idx]
            b_y = target[batch_idx]
            b_w = weights[batch_idx]

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)

            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            batch_weight = float(b_w.sum().item())
            loss_numerator += loss.item() * batch_weight
            weight_sum += batch_weight

    return loss_numerator / max(weight_sum, 1e-12)

In [23]:
# ==========================================
# 所有 MDN 共用的 Early-stopping 训练
# ==========================================
def train_mdn(
    model, X_train_macro, X_train_micro, X_train_industry, y_train, w_train, X_val_macro, 
    X_val_micro, X_val_industry, y_val, w_val, epochs=300, batch_size=1024, lr=1e-3, patience=20,
):
    if run_test == "quick":
        epochs = 2
        patience = 1

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4,)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
        mode="min", factor=0.5, patience=5, threshold=1e-4, threshold_mode="abs", min_lr=1e-6,
    )

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            len(y_train), batch_size, X_train_macro.device, shuffle=True,
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx]
            b_w = w_train[batch_idx]

            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro,b_industry,)

            loss = mdn_nll_loss_weighted(
                pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0,)
            optimizer.step()

            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight

        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)

        val_loss = evaluate_mdn(
            model, X_val_macro, X_val_micro, X_val_industry, y_val, w_val, batch_size,
        )
        if not np.isfinite(val_loss):
            raise RuntimeError("Validation NLL became non-finite.")

        val_loss_history.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    return model, best_val_loss, train_loss_history, val_loss_history

In [24]:
# ==========================================
# 所有 MDN 共用的滚动样本外预测
# ==========================================
def run_mdn_model(panel, model_class, model_name, num_components, 
                  dist_type="normal", model_kwargs=None,
):
    
    panel = panel.sort_values(["mthcaldt", "permno"]).reset_index(drop=True)

    model_kwargs = {} if model_kwargs is None else model_kwargs
    window_gen = make_window_generator(panel)
    global_X_mac, global_X_mic, global_ind, global_y = make_device_tensors(panel)

    all_predictions = []
    history_logs = []
    embedding_history = []

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[{model_name} K={num_components} | Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )

        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_y[idx_tr]
        X_va_mic, X_va_ind, y_va = global_X_mic[idx_va], global_ind[idx_va], global_y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]

        w_tr = torch.from_numpy(
            calculate_sample_weights(train_df["mthcaldt"], mode=weights_type, half_life_months=36)
        ).to(device)

        w_va = torch.from_numpy(
            calculate_sample_weights(val_df["mthcaldt"], mode=weights_type, half_life_months=36)
        ).to(device)

        # 每个窗口独立初始化，不使用 warm start
        seed_everything(random_seed + window_idx)

        model = model_class(macro_dim=len(macro_cols), micro_dim=len(micro_cols), num_industries=50,
            num_components=num_components, dropout_rate=dropout_rate, dist_type=dist_type, **model_kwargs,
        ).to(device)

        model, best_val_loss, tr_hist, va_hist = train_mdn(
            model=model, X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, X_train_industry=X_tr_ind,
            y_train=y_tr, w_train=w_tr, X_val_macro=X_va_mac, X_val_micro=X_va_mic, X_val_industry=X_va_ind,
            y_val=y_va, w_val=w_va, epochs=early_stopping_max_epochs, batch_size=batch_size, lr=lr,
            patience=patience)

        model.eval()

        with torch.no_grad():
            pi_logits, mu, sigma, nu = model(X_te_mac, X_te_mic, X_te_ind)
            pi = torch.softmax(pi_logits, dim=1)

        output = test_df[["permno", "mthcaldt", "target_ret_final"]].copy()

        output["model_id"] = f"{model_name}_{dist_type}_k{num_components}"
        output["pi_vec"] = [json.dumps(row.tolist()) for row in pi.cpu().numpy()]
        output["pi_logits_vec"] = [json.dumps(row.tolist()) for row in pi_logits.cpu().numpy()]
        output["mu_vec"] = [json.dumps(row.tolist()) for row in mu.cpu().numpy()]
        output["sigma_vec"] = [json.dumps(row.tolist()) for row in sigma.cpu().numpy()]
        output["nu_vec"] = [json.dumps(row.tolist()) for row in nu.cpu().numpy()] if nu is not None else None

        all_predictions.append(output)

        embedding_weights = model.ind_embedding.weight.detach().cpu().numpy()
        window_embeddings = pd.DataFrame(
            embedding_weights, columns=[f"emb_dim_{i + 1}" for i in range(embedding_weights.shape[1])]
        )
        window_embeddings["ffi49"] = window_embeddings.index
        window_embeddings["window"] = window_idx
        window_embeddings["test_period"] = f"{info['test'][0]} to {info['test'][1]}"
        embedding_history.append(window_embeddings)

        history_logs.append({
            "window": window_idx,
            "model_id": f"{model_name}_{dist_type}_k{num_components}",
            "train_period": f"{info['train'][0]} to {info['train'][1]}",
            "test_period": f"{info['test'][0]} to {info['test'][1]}",
            "duration_sec": time.time() - start_time,
            "best_val_loss": best_val_loss,
            "best_epoch": int(np.argmin(va_hist) + 1),
            "epochs_run": len(tr_hist),
            "train_loss_trace": tr_hist,
            "val_loss_trace": va_hist,
        })

        if run_test == "quick" and window_idx == 1:
            break

    forecasts = pd.concat(all_predictions, ignore_index=True)
    metrics = pd.DataFrame(history_logs)
    embeddings = pd.concat(embedding_history, ignore_index=True)
    return forecasts, metrics, embeddings

In [ ]:
fused_k1_forecasts, fused_k1_metrics, fused_k1_embeddings = run_mdn_model(
    panel=processed_df,
    model_class=FusedMDN,
    model_name="fused",
    num_components=1,
    dist_type="normal",
    model_kwargs={"ind_emb_dim": 6, "hidden_dim": 64, "head_dim": 8},
)
# 执行保存
# fused_k1_forecasts.to_parquet(os.path.join(path, f'fused_k1_forecasts_{timestamp}.parquet'), index=False)
fused_k1_metrics.to_parquet(os.path.join(path, f'fused_k1_metrics_{timestamp}.parquet'), index=False)
fused_k1_embeddings.to_parquet(os.path.join(path, f'fused_k1_embeddings_{timestamp}.parquet'), index=False)


display(pd.concat([fused_k1_forecasts.head(4), fused_k1_forecasts.tail(4),]))
display(pd.concat([fused_k1_metrics.head(2), fused_k1_metrics.tail(2),]))
display(pd.concat([fused_k1_embeddings.head(4), fused_k1_embeddings.tail(4),]))

[fused K=1 | Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[fused K=1 | Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k1,[1.0],[0.0],[-0.020475909113883972],[0.10932008177042007],None
1,10002,2007-01-31,0.033054,fused_normal_k1,[1.0],[0.0],[-0.007578816264867783],[0.1214563399553299],None
2,10025,2007-01-31,-0.028094,fused_normal_k1,[1.0],[0.0],[0.02010113000869751],[0.17241276800632477],None
3,10026,2007-01-31,-0.038517,fused_normal_k1,[1.0],[0.0],[0.004324318841099739],[0.09342116117477417],None
87371,92869,2008-12-31,-0.366006,fused_normal_k1,[1.0],[0.0],[0.03387029469013214],[0.13349907100200653],None
87372,92872,2008-12-31,-0.175000,fused_normal_k1,[1.0],[0.0],[0.03746585547924042],[0.1313379853963852],None
87373,92874,2008-12-31,-0.496259,fused_normal_k1,[1.0],[0.0],[0.04493873193860054],[0.10880593210458755],None
87374,93105,2008-12-31,0.106070,fused_normal_k1,[1.0],[0.0],[0.06352086365222931],[0.13963189721107483],None


,window,model_id,train_period,test_period,duration_sec,best_val_loss,best_epoch,epochs_run,train_loss_trace,val_loss_trace
0,0,fused_normal_k1,1991-01 to 2005-12,2007-01 to 2007-12,18.681223,-0.875583,1,2,"[-0.712729545910763, -0.7542988721753939]","[-0.8755834553678125, -0.8450201165441023]"
1,1,fused_normal_k1,1992-01 to 2006-12,2008-01 to 2008-12,18.589073,-0.839930,2,2,"[-0.7298498886056185, -0.767634530452412]","[-0.8385463858947755, -0.8399300389987615]"
0,0,fused_normal_k1,1991-01 to 2005-12,2007-01 to 2007-12,18.681223,-0.875583,1,2,"[-0.712729545910763, -0.7542988721753939]","[-0.8755834553678125, -0.8450201165441023]"
1,1,fused_normal_k1,1992-01 to 2006-12,2008-01 to 2008-12,18.589073,-0.839930,2,2,"[-0.7298498886056185, -0.767634530452412]","[-0.8385463858947755, -0.8399300389987615]"


,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,ffi49,window,test_period
0,-0.195762,0.063197,0.174365,-0.048281,-0.098793,-0.065362,0,0,2007-01 to 2007-12
1,-0.054514,0.009263,0.108626,-0.160861,-0.280443,-0.158603,1,0,2007-01 to 2007-12
2,-0.057823,0.086130,0.089509,0.002991,-0.210614,-0.088230,2,0,2007-01 to 2007-12
3,-0.195918,0.214893,0.187753,-0.001165,-0.177425,0.152630,3,0,2007-01 to 2007-12
96,-0.030147,0.002627,0.035868,-0.148194,0.095165,0.187566,46,1,2008-01 to 2008-12
97,0.211070,0.204921,0.003917,0.113762,-0.001734,0.446187,47,1,2008-01 to 2008-12
98,0.082511,0.118154,0.162863,0.341949,-0.034927,-0.161056,48,1,2008-01 to 2008-12
99,-0.109945,-0.089989,-0.117051,-0.207556,0.093466,-0.058321,49,1,2008-01 to 2008-12


In [ ]:
fused_k3_forecasts, fused_k3_metrics, fused_k3_embeddings = run_mdn_model(
    processed_df,
    FusedMDN,
    "fused",
    num_components=3,
    dist_type="normal",
    model_kwargs={"ind_emb_dim": 6, "hidden_dim": 64, "head_dim": 8},
)

# 执行保存
# fused_k3_forecasts.to_parquet(os.path.join(path, f'fused_k3_forecasts_{timestamp}.parquet'), index=False)
fused_k3_metrics.to_parquet(os.path.join(path, f'fused_k3_metrics_{timestamp}.parquet'), index=False)
fused_k3_embeddings.to_parquet(os.path.join(path, f'fused_k3_embeddings_{timestamp}.parquet'), index=False)

display(pd.concat([fused_k3_forecasts.head(4), fused_k3_forecasts.tail(4),]))
display(pd.concat([fused_k3_metrics.head(2), fused_k3_metrics.tail(2),]))
display(pd.concat([fused_k3_embeddings.head(4), fused_k3_embeddings.tail(4),]))

[fused K=3 | Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[fused K=3 | Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k3,"[0.01860729046165943, 0.37179774045944214, 0.6...","[-2.2584547996520996, 0.7363414764404297, 1.23...","[0.19245000183582306, 0.01502302661538124, -0....","[0.28338998556137085, 0.11036743223667145, 0.0...",None
1,10002,2007-01-31,0.033054,fused_normal_k3,"[0.018861426040530205, 0.42751795053482056, 0....","[-2.2571771144866943, 0.8637003898620605, 1.12...","[0.12915487587451935, 0.021055761724710464, -0...","[0.26866835355758667, 0.11155590415000916, 0.0...",None
2,10025,2007-01-31,-0.028094,fused_normal_k3,"[0.02917122095823288, 0.5422647595405579, 0.42...","[-1.9064762592315674, 1.0160952806472778, 0.78...","[0.139872208237648, 0.04362967237830162, -0.01...","[0.31524908542633057, 0.14201955497264862, 0.0...",None
3,10026,2007-01-31,-0.038517,fused_normal_k3,"[0.017596077173948288, 0.39967378973960876, 0....","[-2.3117330074310303, 0.8112396597862244, 1.18...","[0.08172918111085892, 0.023852165788412094, -0...","[0.2407369166612625, 0.10393106937408447, 0.05...",None
87371,92869,2008-12-31,-0.366006,fused_normal_k3,"[0.03399570658802986, 0.6413760185241699, 0.32...","[-1.567887544631958, 1.3694939613342285, 0.688...","[0.12053556740283966, 0.036757342517375946, 0....","[0.29783597588539124, 0.12982343137264252, 0.0...",None
87372,92872,2008-12-31,-0.175000,fused_normal_k3,"[0.03715025633573532, 0.6596784591674805, 0.30...","[-1.5002317428588867, 1.3765499591827393, 0.59...","[0.12328237295150757, 0.03611171990633011, -0....","[0.3057815730571747, 0.1389371156692505, 0.069...",None
87373,92874,2008-12-31,-0.496259,fused_normal_k3,"[0.03375515341758728, 0.6423104405403137, 0.32...","[-1.5700511932373047, 1.375887393951416, 0.691...","[0.12229332327842712, 0.04189710319042206, 0.0...","[0.26920458674430847, 0.11722017824649811, 0.0...",None
87374,93105,2008-12-31,0.106070,fused_normal_k3,"[0.03968963027000427, 0.6522449851036072, 0.30...","[-1.4637811183929443, 1.335548996925354, 0.585...","[0.20258048176765442, 0.0573614165186882, 0.00...","[0.35769641399383545, 0.16119642555713654, 0.0...",None


,window,model_id,train_period,test_period,duration_sec,best_val_loss,best_epoch,epochs_run,train_loss_trace,val_loss_trace
0,0,fused_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,21.936755,-0.979214,2,2,"[-0.7843590576638743, -0.831620631954807]","[-0.9159003461697792, -0.9792137331386018]"
1,1,fused_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,22.619697,-0.881913,1,2,"[-0.803634812930807, -0.8487687977142058]","[-0.8819130594061155, -0.8702828483256845]"
0,0,fused_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,21.936755,-0.979214,2,2,"[-0.7843590576638743, -0.831620631954807]","[-0.9159003461697792, -0.9792137331386018]"
1,1,fused_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,22.619697,-0.881913,1,2,"[-0.803634812930807, -0.8487687977142058]","[-0.8819130594061155, -0.8702828483256845]"


,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,ffi49,window,test_period
0,-0.234947,-0.100178,-0.004214,0.302064,-0.278646,0.059468,0,0,2007-01 to 2007-12
1,0.188046,-0.362611,0.029253,0.160656,-0.110957,-0.029149,1,0,2007-01 to 2007-12
2,0.217955,-0.387223,0.192631,0.032693,-0.172469,-0.030281,2,0,2007-01 to 2007-12
3,0.161133,-0.158315,0.087140,0.135702,-0.137650,0.227460,3,0,2007-01 to 2007-12
96,-0.076817,-0.141138,0.020513,0.020408,0.129708,0.184708,46,1,2008-01 to 2008-12
97,0.067274,-0.237597,0.136778,-0.058906,0.186915,0.257672,47,1,2008-01 to 2008-12
98,0.123087,0.177843,0.159702,-0.074517,0.091682,0.059755,48,1,2008-01 to 2008-12
99,-0.130074,-0.272486,-0.117793,0.101153,-0.081477,-0.073152,49,1,2008-01 to 2008-12


### ==========================================
### Structure K
### ==========================================

In [27]:
# ==========================================
# Structured MDN：Macro → pi，Micro + Industry → mu 和 sigma
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, macro_hidden_dim=32,
                 micro_hidden_dim=64, head_dim=8, num_components=3, dropout_rate=0.05,
                 dist_type="normal"):
        super().__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        self.ind_embedding = nn.Embedding(num_industries, ind_emb_dim)

        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim, macro_hidden_dim // 2),
            nn.LayerNorm(macro_hidden_dim // 2),
            nn.ELU(),
        )

        self.micro_net = nn.Sequential(
            nn.Linear(micro_dim + ind_emb_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(micro_hidden_dim // 2, head_dim),
            nn.LayerNorm(head_dim),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(macro_hidden_dim // 2, num_components) if num_components > 1 else None
        self.mu_head = nn.Linear(head_dim, num_components)
        self.sigma_head = nn.Linear(head_dim, num_components)
        self.nu_head = nn.Linear(head_dim, num_components) if dist_type == "t" else None

        nn.init.normal_(self.ind_embedding.weight, mean=0.0, std=0.05)
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)
        nn.init.xavier_normal_(self.sigma_head.weight, gain=0.1)
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.pi_head is not None:
            nn.init.zeros_(self.pi_head.weight)
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            nn.init.xavier_normal_(self.nu_head.weight, gain=0.1)
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_industry):
        h_macro = self.macro_net(x_macro)

        if self.pi_head is None:
            pi_logits = h_macro.new_zeros((len(h_macro), 1))
        else:
            pi_logits = self.pi_head(h_macro)

        industry_embedding = self.ind_embedding(x_industry)
        h_micro = self.micro_net(torch.cat([x_micro, industry_embedding], dim=1))

        mu = self.mu_head(h_micro)
        sigma = F.softplus(self.sigma_head(h_micro)) + sigma_floor
        nu = F.softplus(self.nu_head(h_micro)) + 2.01 if self.nu_head is not None else None

        return pi_logits, mu, sigma, nu

In [28]:
# ==========================================
# 运行 Structured MDN
# ==========================================
def run_structured_model(panel, num_components=3, dist_type="normal"):
    model_args = {
        "ind_emb_dim": 6,
        "macro_hidden_dim": 16,
        "micro_hidden_dim": 64,
        "head_dim": 8,
    }

    return run_mdn_model(
        panel, StructuredMDN, "structured", num_components, dist_type, model_args
    )

In [ ]:
structured_forecasts, structured_metrics, structured_embeddings = run_structured_model(
    processed_df, num_components=3, dist_type="normal"
)

# 执行保存
# structured_forecasts.to_parquet(os.path.join(path, f'structured_forecasts_{timestamp}.parquet'), index=False)
structured_metrics.to_parquet(os.path.join(path, f'structured_metrics_{timestamp}.parquet'), index=False)
structured_embeddings.to_parquet(os.path.join(path, f'structured_embeddings_{timestamp}.parquet'), index=False)

display(pd.concat([structured_forecasts.head(4), structured_forecasts.tail(4)]))
display(pd.concat([structured_metrics.head(2), structured_metrics.tail(2)]))
display(pd.concat([structured_embeddings.head(4), structured_embeddings.tail(4)]))

[structured K=3 | Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[structured K=3 | Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,structured_normal_k3,"[0.017875492572784424, 0.3509249687194824, 0.6...","[-2.3612847328186035, 0.6158570051193237, 1.20...","[0.16704817116260529, 0.03774138540029526, -0....","[0.3483477532863617, 0.11937367171049118, 0.05...",None
1,10002,2007-01-31,0.033054,structured_normal_k3,"[0.017875492572784424, 0.3509249687194824, 0.6...","[-2.3612847328186035, 0.6158570051193237, 1.20...","[0.06662852317094803, 0.05911277234554291, 0.0...","[0.337143212556839, 0.15371887385845184, 0.074...",None
2,10025,2007-01-31,-0.028094,structured_normal_k3,"[0.017875492572784424, 0.3509249687194824, 0.6...","[-2.3612847328186035, 0.6158570051193237, 1.20...","[0.1448555290699005, 0.05752973258495331, -0.0...","[0.4642188549041748, 0.21735665202140808, 0.10...",None
3,10026,2007-01-31,-0.038517,structured_normal_k3,"[0.017875492572784424, 0.3509249687194824, 0.6...","[-2.3612847328186035, 0.6158570051193237, 1.20...","[0.0660385712981224, 0.0417025052011013, 0.000...","[0.2954486310482025, 0.12641026079654694, 0.06...",None
87371,92869,2008-12-31,-0.366006,structured_normal_k3,"[0.4055947959423065, 0.18294252455234528, 0.41...","[0.2456185668706894, -0.5505639910697937, 0.25...","[-0.0693986788392067, 0.014450205489993095, 0....","[0.10460884869098663, 0.4191187024116516, 0.15...",None
87372,92872,2008-12-31,-0.175000,structured_normal_k3,"[0.4055947959423065, 0.18294252455234528, 0.41...","[0.2456185668706894, -0.5505639910697937, 0.25...","[-0.06089317053556442, 0.0108173917979002, 0.0...","[0.10638611763715744, 0.3619733154773712, 0.13...",None
87373,92874,2008-12-31,-0.496259,structured_normal_k3,"[0.4055947959423065, 0.18294252455234528, 0.41...","[0.2456185668706894, -0.5505639910697937, 0.25...","[0.002999657765030861, 0.06802920997142792, 0....","[0.04971599951386452, 0.2229807823896408, 0.08...",None
87374,93105,2008-12-31,0.106070,structured_normal_k3,"[0.4055947959423065, 0.18294252455234528, 0.41...","[0.2456185668706894, -0.5505639910697937, 0.25...","[-0.04272277653217316, 0.04989507049322128, 0....","[0.10158092528581619, 0.35109949111938477, 0.1...",None


,window,model_id,train_period,test_period,duration_sec,best_val_loss,best_epoch,epochs_run,train_loss_trace,val_loss_trace
0,0,structured_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,33.882412,-1.044987,1,2,"[-0.7580800615586929, -0.7816136201365964]","[-1.0449874617336394, -1.035170717232201]"
1,1,structured_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,33.524508,-0.846060,2,2,"[-0.7823938954756146, -0.8164110618918146]","[-0.8380687281945832, -0.8460600980209444]"
0,0,structured_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,33.882412,-1.044987,1,2,"[-0.7580800615586929, -0.7816136201365964]","[-1.0449874617336394, -1.035170717232201]"
1,1,structured_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,33.524508,-0.846060,2,2,"[-0.7823938954756146, -0.8164110618918146]","[-0.8380687281945832, -0.8460600980209444]"


,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,ffi49,window,test_period
0,0.001623,0.056809,-0.315578,-0.152923,-0.091114,0.182191,0,0,2007-01 to 2007-12
1,0.384348,0.058124,0.057838,-0.202716,-0.015153,0.150171,1,0,2007-01 to 2007-12
2,0.208653,-0.169913,0.172618,-0.098956,-0.090328,0.096922,2,0,2007-01 to 2007-12
3,0.098218,-0.195818,0.110270,-0.170404,-0.071865,0.156578,3,0,2007-01 to 2007-12
96,-0.039503,-0.130706,-0.081971,-0.180800,-0.089181,0.112630,46,1,2008-01 to 2008-12
97,0.047856,-0.388130,-0.087170,-0.123151,0.077499,0.302488,47,1,2008-01 to 2008-12
98,-0.071052,-0.008365,0.122390,-0.165642,0.284162,0.088630,48,1,2008-01 to 2008-12
99,0.110270,-0.204342,-0.098039,0.167131,-0.331559,-0.056291,49,1,2008-01 to 2008-12


In [30]:
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output


In [31]:
# ==========================================
# 根据 MDN 预测计算 VaR 和 ES
# ==========================================
def calculate_es_table(forecasts, alphas=(0.01, 0.05, 0.10),
                       chunk_size=50_000, max_rows=None, iterations=45):

    """
    根据一个模型的 MDN forecast DataFrame 计算左尾 VaR 和 loss-ES。

    var_alpha = F^{-1}(alpha)，是收益率分位数；
    es_alpha  = -E[R | R <= var_alpha]，是带符号的 loss-ES。

    每次只能传入一个 model_id，但可以是 K=1、K=3、Normal 或 Student-t。
    """
    frame = forecasts.iloc[:max_rows] if max_rows is not None else forecasts
    alphas = np.asarray(alphas, dtype=float)

    def decode(column):
        values = frame[column].map(lambda x: json.loads(x) if isinstance(x, str) else x)
        return np.stack(values).astype(float, copy=False)

    pi = decode("pi_vec")
    mu = decode("mu_vec")
    sigma = decode("sigma_vec")

    if pi.shape != mu.shape or mu.shape != sigma.shape:
        raise ValueError("pi、mu 和 sigma 的维度不一致。")
    if not np.isfinite(pi).all() or not np.isfinite(mu).all() or not np.isfinite(sigma).all():
        raise ValueError("pi、mu 或 sigma 含有 NaN/inf。")
    if np.any(sigma <= 0):
        raise ValueError("sigma 必须大于0。")
    if not np.allclose(pi.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError("部分 mixture probabilities 加总不等于1。")

    # Normal 模型的 nu_vec 全为空；Student-t 模型的 nu_vec 应全部非空
    nu = None
    if "nu_vec" in frame.columns:
        has_nu = frame["nu_vec"].notna()

        if has_nu.any() and not has_nu.all():
            raise ValueError("nu_vec 不能同时包含空值和非空值。")

        if has_nu.all():
            nu = decode("nu_vec")
            if nu.shape != mu.shape or not np.isfinite(nu).all() or np.any(nu <= 2):
                raise ValueError("nu 必须与 mu 维度一致，而且全部大于2。")
            
    n_rows, n_components = pi.shape
    n_alphas = len(alphas)

    var_matrix = np.empty((n_rows, n_alphas), dtype=float)
    es_matrix = np.empty((n_rows, n_alphas), dtype=float)
    bracket_probability = max(float(alphas.min()) * 0.1, 1e-10)

    for start in range(0, n_rows, chunk_size):
        stop = min(start + chunk_size, n_rows)

        p = pi[start:stop]
        m = mu[start:stop]
        s = sigma[start:stop]
        degrees = None if nu is None else nu[start:stop]

        # Normal K=1 有闭式解，不需要二分法
        if degrees is None and n_components == 1:
            z = norm.ppf(alphas)[None, :]
            chunk_var = m + s * z
            chunk_es = -m + s * norm.pdf(z) / alphas[None, :]

        else:
            # 为每一行建立足够宽的分位数搜索区间
            if degrees is None:
                lower_component = m + s * norm.ppf(bracket_probability)
                upper_component = m + s * norm.ppf(1.0 - bracket_probability)
            else:
                lower_component = m + s * student_t.ppf(bracket_probability, df=degrees)
                upper_component = m + s * student_t.ppf(1.0 - bracket_probability, df=degrees)

            lower = np.repeat(lower_component.min(axis=1)[:, None], n_alphas, axis=1)
            upper = np.repeat(upper_component.max(axis=1)[:, None], n_alphas, axis=1)

            # 批量二分法求 mixture quantiles
            for _ in range(iterations):
                middle = (lower + upper) / 2.0
                z = (middle[..., None] - m[:, None, :]) / s[:, None, :]

                if degrees is None:
                    component_cdf = norm.cdf(z)
                else:
                    component_cdf = student_t.cdf(z, df=degrees[:, None, :])

                mixture_cdf = np.sum(p[:, None, :] * component_cdf, axis=2)
                lower = np.where(mixture_cdf < alphas, middle, lower)
                upper = np.where(mixture_cdf >= alphas, middle, upper)

            chunk_var = (lower + upper) / 2.0
            z = (chunk_var[..., None] - m[:, None, :]) / s[:, None, :]

            if degrees is None:
                component_cdf = norm.cdf(z)
                adjusted_pdf = norm.pdf(z)
            else:
                degrees_3d = degrees[:, None, :]
                component_cdf = student_t.cdf(z, df=degrees_3d)
                adjusted_pdf = (
                    (degrees_3d + z**2) / (degrees_3d - 1.0)
                    * student_t.pdf(z, df=degrees_3d)
                )

            left_moment = (
                m[:, None, :] * component_cdf
                - s[:, None, :] * adjusted_pdf
            )

            chunk_es = (
                -np.sum(p[:, None, :] * left_moment, axis=2)
                / alphas[None, :]
            )

        var_matrix[start:stop] = chunk_var
        es_matrix[start:stop] = chunk_es

    # Mixture conditional mean and standard deviation
    mean_pred = np.sum(pi * mu, axis=1)

    if nu is None:
        component_variance = sigma**2
    else:
        component_variance = sigma**2 * nu / (nu - 2.0)

    mixture_variance = (
        np.sum(pi * (mu**2 + component_variance), axis=1)
        - mean_pred**2
    )
    vol_pred = np.sqrt(np.maximum(mixture_variance, 0.0))

    result = frame[["permno", "mthcaldt", "model_id"]].reset_index(drop=True)

    for column, alpha in enumerate(alphas):
        suffix = f"{alpha * 100:g}".replace(".", "_")
        result[f"es_{suffix}"] = es_matrix[:, column]
        result[f"var_{suffix}"] = var_matrix[:, column]

    result["mean_pred"] = mean_pred
    result["vol_pred"] = vol_pred
    result["realized_ret"] = frame["target_ret_final"].to_numpy(dtype=float)

    return result

In [32]:
# ==========================================
# 计算 VaR/ES，合并回原始 forecasts，再保存
# ==========================================
def attach_var_es(forecasts):
    risk = calculate_es_table(forecasts, alphas=ALPHAS, max_rows=None)

    keys = ["permno", "mthcaldt", "model_id"]
    risk_cols = [
        f"{name}_{ALPHA_SUFFIX[alpha]}"
        for alpha in ALPHAS
        for name in ("var", "es")
    ]

    # 防止重复运行时产生 var_1_x、var_1_y
    # forecasts = forecasts.drop(columns=risk_cols, errors="ignore")

    combined = forecasts.merge(
        risk[keys + risk_cols],
        on=keys,
        how="left",
        validate="one_to_one",
    )

    # if combined[risk_cols].isna().any().any():
    #     raise ValueError("部分 forecasts 没有匹配到 VaR/ES。")

    return combined


max_rows = None

fused_k1_forecasts = attach_var_es(fused_k1_forecasts)
fused_k3_forecasts = attach_var_es(fused_k3_forecasts)
structured_forecasts = attach_var_es(structured_forecasts)

forecast_files = {
    "fused_k1": fused_k1_forecasts,
    "fused_k3": fused_k3_forecasts,
    "structured": structured_forecasts,
}

for name, forecasts in forecast_files.items():
    forecasts.to_parquet(
        os.path.join(path, f"{name}_forecasts_{timestamp}.parquet"),
        index=False,
    )

display(fused_k1_forecasts.head(2))
display(fused_k3_forecasts.head(2))
display(structured_forecasts.head(2))

,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec,var_1,es_1,var_5,es_5,var_10,es_10
0,10001,2007-01-31,0.264774,fused_normal_k1,[1.0],[0.0],[-0.020475909113883972],[0.10932008177042007],None,-0.274792,0.311837,-0.200291,0.245972,-0.160575,0.212331
1,10002,2007-01-31,0.033054,fused_normal_k1,[1.0],[0.0],[-0.007578816264867783],[0.1214563399553299],None,-0.290129,0.331286,-0.207357,0.258108,-0.163231,0.220733


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec,var_1,es_1,var_5,es_5,var_10,es_10
0,10001,2007-01-31,0.264774,fused_normal_k3,"[0.01860729046165943, 0.37179774045944214, 0.6...","[-2.2584547996520996, 0.7363414764404297, 1.23...","[0.19245000183582306, 0.01502302661538124, -0....","[0.28338998556137085, 0.11036743223667145, 0.0...",None,-0.205494,0.259687,-0.121362,0.173510,-0.091003,0.138817
1,10002,2007-01-31,0.033054,fused_normal_k3,"[0.018861426040530205, 0.42751795053482056, 0....","[-2.2571771144866943, 0.8637003898620605, 1.12...","[0.12915487587451935, 0.021055761724710464, -0...","[0.26866835355758667, 0.11155590415000916, 0.0...",None,-0.210930,0.268402,-0.126912,0.179792,-0.094127,0.144137


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec,var_1,es_1,var_5,es_5,var_10,es_10
0,10001,2007-01-31,0.264774,structured_normal_k3,"[0.017875492572784424, 0.3509249687194824, 0.6...","[-2.3612847328186035, 0.6158570051193237, 1.20...","[0.16704817116260529, 0.03774138540029526, -0....","[0.3483477532863617, 0.11937367171049118, 0.05...",None,-0.204972,0.283092,-0.120603,0.176193,-0.091757,0.140306
1,10002,2007-01-31,0.033054,structured_normal_k3,"[0.017875492572784424, 0.3509249687194824, 0.6...","[-2.3612847328186035, 0.6158570051193237, 1.20...","[0.06662852317094803, 0.05911277234554291, 0.0...","[0.337143212556839, 0.15371887385845184, 0.074...",None,-0.258047,0.350760,-0.146935,0.218812,-0.105754,0.171374


In [33]:
# # run_test = "full"  # "quick" 或 "full"
# max_rows = 100 if run_test == "quick" else None
# alphas = (0.01, 0.05, 0.10)

# fused_k1_es = calculate_es_table(
#     fused_k1_forecasts, alphas=alphas, max_rows=max_rows
# )
# fused_k1_es.to_parquet(
#     os.path.join(path, f"ES_fused_normal_k1_{timestamp}.parquet"), index=False
# )

# fused_k3_es = calculate_es_table(
#     fused_k3_forecasts, alphas=alphas, max_rows=max_rows
# )
# fused_k3_es.to_parquet(
#     os.path.join(path, f"ES_fused_normal_k3_{timestamp}.parquet"), index=False
# )

# structured_es = calculate_es_table(
#     structured_forecasts, alphas=alphas, max_rows=max_rows
# )
# structured_es.to_parquet(
#     os.path.join(path, f"ES_structured_normal_k3_{timestamp}.parquet"), index=False
# )

# display(pd.concat([fused_k1_es.head(2), fused_k3_es.tail(2)], ignore_index=True))
# display(pd.concat([structured_es.head(2), structured_es.tail(2)], ignore_index=True))
# display(pd.concat([fused_k1_es.head(2), structured_es.head(2)], ignore_index=True))

In [34]:
def summarize_training(metrics):
    rows = []

    for _, row in metrics.iterrows():
        train_loss = np.asarray(row["train_loss_trace"], dtype=float)
        val_loss = np.asarray(row["val_loss_trace"], dtype=float)
        best_idx = np.argmin(val_loss)

        rows.append({
            "window": row["window"],
            "model_id": row["model_id"],
            "epochs_run": len(val_loss),
            "best_epoch": best_idx + 1,
            "train_first": train_loss[0],
            "train_best_epoch": train_loss[best_idx],
            "train_last": train_loss[-1],
            "val_first": val_loss[0],
            "val_best": val_loss[best_idx],
            "val_last": val_loss[-1],
        })

    return pd.DataFrame(rows)


display(summarize_training(fused_k1_metrics))
display(summarize_training(fused_k3_metrics))
display(summarize_training(structured_metrics))

,window,model_id,epochs_run,best_epoch,train_first,train_best_epoch,train_last,val_first,val_best,val_last
0,0,fused_normal_k1,2,1,-0.71273,-0.712730,-0.754299,-0.875583,-0.875583,-0.84502
1,1,fused_normal_k1,2,2,-0.72985,-0.767635,-0.767635,-0.838546,-0.839930,-0.83993


,window,model_id,epochs_run,best_epoch,train_first,train_best_epoch,train_last,val_first,val_best,val_last
0,0,fused_normal_k3,2,2,-0.784359,-0.831621,-0.831621,-0.915900,-0.979214,-0.979214
1,1,fused_normal_k3,2,1,-0.803635,-0.803635,-0.848769,-0.881913,-0.881913,-0.870283


,window,model_id,epochs_run,best_epoch,train_first,train_best_epoch,train_last,val_first,val_best,val_last
0,0,structured_normal_k3,2,1,-0.758080,-0.758080,-0.781614,-1.044987,-1.044987,-1.035171
1,1,structured_normal_k3,2,2,-0.782394,-0.816411,-0.816411,-0.838069,-0.846060,-0.846060


In [35]:
def mixture_diagnostics(forecasts):
    pi = np.stack(
        forecasts["pi_vec"].map(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
    )

    return pd.Series({
        "mean_effective_k": np.mean(1.0 / np.sum(pi**2, axis=1)),
        "mean_max_pi": np.mean(pi.max(axis=1)),
        "share_max_pi_above_0_95": np.mean(pi.max(axis=1) > 0.95),
    })


display(mixture_diagnostics(fused_k3_forecasts))
display(mixture_diagnostics(structured_forecasts))

mean_effective_k           2.041496
mean_max_pi                0.582306
share_max_pi_above_0_95    0.000000
dtype: float64

mean_effective_k           1.940292
mean_max_pi                0.637464
share_max_pi_above_0_95    0.000000
dtype: float64

In [36]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()